# <font color="brown">Guided Project, Day 3: Recommending the Next Product</font>

## <font color="brown">Problem Statement</font>

### <font color="blue">Context</font>

Day 3, the payoff of the whole 3-day project. Day 1 grouped customers into behavioral segments.
Day 2 scored every customer's churn risk. Today builds a recommendation system that suggests the
next banking product for each customer, and uses both of the earlier days' outputs to answer the
question that actually matters to the business: **for a customer who is both at high risk of
churning and already showing "at-risk" behavior, which product would give the bank the best shot
at retaining them?**

### <font color="blue">Objective</font>

- Build collaborative filtering and content-based recommenders for banking products, from scratch.
- Blend them into a hybrid recommender, and demonstrate the cold-start problem for both a new
  customer and a new product.
- Bring in Day 1's segment and Day 2's churn risk score to target recommendations at the
  customers who need them most.

### <font color="blue">Data Dictionary</font>

| File | Grain | Contents |
| --- | --- | --- |
| `product_catalog.csv` | 1 row / product | 24 banking products, category, fees, rate, risk level |
| `product_holdings.csv` | 1 row / (customer, product) held | engagement score, months held |
| `customer_segments.csv` | 1 row / customer | Day 1's `Customer_Segment` |
| `churn_risk_scores.csv` | 1 row / customer | Day 2's `Churn_Risk_Score` |

### <font color="blue">One Design Change From a Small-Scale Recommender</font>

With 450,000 customers, comparing every customer to every other customer (user-based collaborative
filtering, the approach that works fine for a few hundred users) would mean a 450,000 x 450,000
similarity computation, not practical. **Item-based** collaborative filtering sidesteps this
entirely: with only 24 products, comparing every *product* to every other product is a tiny
24 x 24 computation, regardless of how many customers there are. This is the same reason real
large-scale recommenders ("customers who bought this also bought...") compare items, not people.

## <font color="brown">Importing Necessary Libraries</font>

In [1]:
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

## <font color="brown">Step 1: Reading the Source Files</font>

In [2]:
catalog = pd.read_csv("product_catalog.csv")
holdings = pd.read_csv("product_holdings.csv")
segments = pd.read_csv("customer_segments.csv")
churn_risk = pd.read_csv("churn_risk_scores.csv")

print("Products:", catalog.shape[0])
print("Holdings (customer-product pairs):", holdings.shape[0])
print("Avg products held per customer:", round(holdings.shape[0] / segments.shape[0], 2))
catalog.head()

Products: 24
Holdings (customer-product pairs): 1398186
Avg products held per customer: 3.11


,Product_ID,Product_Name,Category,Min_Balance_Required,Annual_Fee,Interest_Rate_Pct,Risk_Level,Target_Income_Tier
0,PROD_000,Personal Loan,Loan,0,0,11.5,Medium,Mid
1,PROD_001,Auto Loan,Loan,0,0,7.2,Low,Mid
2,PROD_002,Home Mortgage,Loan,0,0,6.5,Low,High
3,PROD_003,Student Loan,Loan,0,0,5.8,Low,Low
4,PROD_004,Small Business Loan,Loan,5000,0,9.0,Medium,High


## <font color="brown">Step 2: Building the Customer-Product Matrix</font>

The same reshape as Day 1's ratings matrix: one row per customer, one column per product, the
cell is that customer's engagement score (0 if they don't hold that product at all).

In [3]:
customer_product_matrix = holdings.pivot_table(
    index="Customer_ID", columns="Product_ID", values="Engagement_Score", fill_value=0
)

print(customer_product_matrix.shape)
customer_product_matrix.iloc[:5, :6]

(450000, 24)


Product_ID,PROD_000,PROD_001,PROD_002,PROD_003,PROD_004,PROD_005
Customer_ID,,,,,,
CUST_0000000,61.3,0.0,0.0,0.0,0.0,0.0
CUST_0000001,67.4,0.0,0.0,0.0,0.0,0.0
CUST_0000002,0.0,0.0,0.0,0.0,0.0,0.0
CUST_0000003,0.0,0.0,0.0,0.0,0.0,0.0
CUST_0000004,71.7,0.0,0.0,0.0,0.0,53.6


## <font color="brown">Step 3: Item-Based Collaborative Filtering</font>

Transpose the matrix (products as rows now) and run the same cosine similarity used throughout
this project, this time between *products*, based on which customers engage with them similarly.
Two products end up "similar" if the same kinds of customers hold and engage with both, no product
features involved at all.

In [4]:
item_similarity = cosine_similarity(customer_product_matrix.T)
item_similarity_df = pd.DataFrame(
    item_similarity, index=customer_product_matrix.columns, columns=customer_product_matrix.columns
)

# sanity check: what looks most similar to a Mutual Fund, purely from co-engagement patterns?
sample_product = catalog.loc[catalog["Product_Name"] == "Mutual Fund - Growth", "Product_ID"].iloc[0]
most_similar = item_similarity_df[sample_product].sort_values(ascending=False).iloc[1:6]
catalog.set_index("Product_ID").loc[most_similar.index, ["Product_Name", "Category"]]

,Product_Name,Category
Product_ID,,
PROD_021,Brokerage Account,Investment
PROD_020,Mutual Fund - Balanced,Investment
PROD_022,Retirement Account (IRA),Investment
PROD_023,Robo-Advisor Portfolio,Investment
PROD_016,Auto Insurance,Insurance


**All four other Investment products, plus one Insurance product, computed purely from
co-engagement patterns, no product features involved at all.** Customers who engage with one
investment product tend to engage with others, exactly the kind of structure item-based CF is
supposed to find, and a useful sanity check before trusting it on individual customers.

In [5]:
def cf_recommend(customer_id, top_n=5):
    customer_row = customer_product_matrix.loc[customer_id]
    held_products = customer_row[customer_row > 0].index

    # for every product, score = sum of (this customer's engagement in a held product x
    # how similar that held product is to the candidate product)
    scores = item_similarity_df[held_products].mul(customer_row[held_products], axis=1).sum(axis=1)
    scores = scores.drop(index=held_products, errors="ignore")
    return scores.sort_values(ascending=False).head(top_n)

example_customer = "CUST_0214294"
cf_recommend(example_customer)

Product_ID
PROD_006    5.584834
PROD_005    5.537466
PROD_009    5.531893
PROD_008    5.489437
PROD_007    5.438493
dtype: float64

This customer holds exactly one product, a Home Mortgage. Item-based CF's answer: **the five
credit cards**, other customers who also hold a mortgage tend to hold a credit card too, a
believable, real-world pattern the model found on its own.

## <font color="brown">Step 4: Content-Based Filtering</font>

The product catalog's own features this time, category, risk level, and target income tier, each
one-hot encoded into a feature vector per product.

In [6]:
category_features = pd.get_dummies(catalog["Category"], prefix="Cat")
risk_features = pd.get_dummies(catalog["Risk_Level"], prefix="Risk")
income_features = pd.get_dummies(catalog["Target_Income_Tier"], prefix="Income")

product_features = pd.concat([category_features, risk_features, income_features], axis=1)
product_features.index = catalog["Product_ID"]
product_features = product_features.astype(float)

product_features.head()

,Cat_Card,Cat_Insurance,Cat_Investment,Cat_Loan,Cat_Savings,Risk_High,Risk_Low,Risk_Medium,Income_High,Income_Low,Income_Mid
Product_ID,,,,,,,,,,,
PROD_000,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
PROD_001,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
PROD_002,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
PROD_003,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
PROD_004,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0


In [7]:
def cb_recommend(customer_id, top_n=5):
    customer_row = customer_product_matrix.loc[customer_id]
    held_products = customer_row[customer_row > 0].index

    weights = customer_row[held_products]
    profile = product_features.loc[held_products].mul(weights, axis=0).sum(axis=0) / weights.sum()

    similarity = cosine_similarity(profile.values.reshape(1, -1), product_features.values)[0]
    scores = pd.Series(similarity, index=product_features.index)
    scores = scores.drop(index=held_products, errors="ignore")
    return scores.sort_values(ascending=False).head(top_n)

cb_recommend(example_customer)

Product_ID
PROD_001    0.666667
PROD_003    0.666667
PROD_004    0.666667
PROD_013    0.666667
PROD_014    0.666667
dtype: float64

**A completely different answer: other Loan and Savings products**, not a single credit card in
sight. With only one product to build a profile from, content-based filtering can only lean on
that one product's own features (low risk, high income tier), and finds other low-risk products
aimed at similar customers. CF and CB genuinely disagree here, exactly the situation a hybrid
blend exists for, a customer with a thin history is where these two approaches are most likely to
diverge, not agree.

## <font color="brown">Step 5: The Cold-Start Problem, Again</font>

Same two cases as the earlier recommendation-system topic in this repo, now at bank-customer
scale: a brand-new customer with no products yet, and a brand-new product nobody has adopted yet.

In [8]:
new_customer_row = pd.Series(0, index=customer_product_matrix.columns)
new_customer_similarity_to_products = item_similarity_df.mul(new_customer_row, axis=1).sum(axis=1)

print("Item-based CF score for every product, brand-new customer:")
print(new_customer_similarity_to_products.abs().sum())

Item-based CF score for every product, brand-new customer:
0.0


**Exactly 0.0 for every single product.** With no products held, `new_customer_row` is all
zeros, multiplying it against any similarity row still gives zero, there is nothing for item-based
CF to work from. Structurally identical to the collaborative-filtering cold-start problem seen
earlier in this repo, whether the "items" are movies or banking products, or the technique is
user-based or item-based CF, no history means no recommendation.

In [9]:
# a new product just added to the catalog, category Loan, nobody has adopted it yet
new_product_features = pd.Series(0.0, index=product_features.columns)
new_product_features["Cat_Loan"] = 1.0
new_product_features["Risk_Low"] = 1.0
new_product_features["Income_Mid"] = 1.0

example_profile_weights = customer_product_matrix.loc[example_customer]
example_held = example_profile_weights[example_profile_weights > 0].index
example_profile = product_features.loc[example_held].mul(example_profile_weights[example_held], axis=0).sum(axis=0) / example_profile_weights[example_held].sum()

new_product_score = cosine_similarity(example_profile.values.reshape(1, -1), new_product_features.values.reshape(1, -1))[0, 0]
print(f"Content-based score for the brand-new product: {new_product_score:.3f}")
print("Item-based CF score for the same brand-new product: undefined, zero customers have engaged with it yet")

Content-based score for the brand-new product: 0.667
Item-based CF score for the same brand-new product: undefined, zero customers have engaged with it yet


**Content-based gives a real, defined score (0.667) for a product that has never been held by
anyone.** It needed nothing but the new product's own catalog features (Loan, low risk, mid
income) compared against this customer's existing profile. Item-based CF, by contrast, cannot
compute anything for this product at all, its similarity to every other product depends entirely
on customer engagement data that does not exist yet for something brand new. This is the same
asymmetry seen for the new customer above, just mirrored: content-based filtering solves both
cold-start cases, because it never depended on interaction history in the first place.

## <font color="brown">Step 6: A Hybrid Score, Blended Across All Customers</font>

In [10]:
def min_max_scale(scores):
    return (scores - scores.min()) / (scores.max() - scores.min())

def hybrid_recommend(customer_id, top_n=5, cf_weight=0.5):
    cf_scores = cf_recommend(customer_id, top_n=len(product_features))
    cb_scores = cb_recommend(customer_id, top_n=len(product_features))

    candidates = cf_scores.index.union(cb_scores.index)
    cf_full = cf_scores.reindex(candidates)
    cb_full = cb_scores.reindex(candidates)

    cf_scaled = min_max_scale(cf_full.fillna(cf_full.min()))
    cb_scaled = min_max_scale(cb_full.fillna(cb_full.min()))

    blended = cf_weight * cf_scaled + (1 - cf_weight) * cb_scaled
    return blended.sort_values(ascending=False).head(top_n)

hybrid_recommend(example_customer)

Product_ID
PROD_009    0.720811
PROD_004    0.703714
PROD_008    0.697403
PROD_001    0.683980
PROD_003    0.681430
dtype: float64

## <font color="brown">Step 7: The Payoff, Targeting High-Risk Customers</font>

Everything comes together here: filter to customers who are both in the "At-Risk Declining
Activity" segment (Day 1) **and** flagged high-risk by the churn model (Day 2), and generate
hybrid recommendations for a sample of them.

In [11]:
customer_context = segments.merge(churn_risk, on="Customer_ID")
priority_customers = customer_context[
    (customer_context["Customer_Segment"] == "At-Risk Declining Activity")
    & (customer_context["High_Risk_Flag"] == 1)
]

print("Customers who are both At-Risk segment AND high churn risk:", priority_customers.shape[0])
priority_customers.sort_values("Churn_Risk_Score", ascending=False).head()

Customers who are both At-Risk segment AND high churn risk: 24678


,Customer_ID,Cluster,Customer_Segment,Churn_Risk_Score,High_Risk_Flag
214294,CUST_0214294,3,At-Risk Declining Activity,0.927002,1
101511,CUST_0101511,3,At-Risk Declining Activity,0.922860,1
370674,CUST_0370674,3,At-Risk Declining Activity,0.920749,1
78600,CUST_0078600,3,At-Risk Declining Activity,0.920546,1
280178,CUST_0280178,3,At-Risk Declining Activity,0.917660,1


**24,678 customers, 5.5% of the entire customer base**, sit at the intersection of both earlier
days' work: independently flagged as behaviorally at-risk (Day 1) and as high churn risk (Day 2).
This is the group a retention campaign would realistically target first.

In [12]:
sample_priority_ids = priority_customers.sort_values("Churn_Risk_Score", ascending=False)["Customer_ID"].head(5)

for cid in sample_priority_ids:
    recs = hybrid_recommend(cid, top_n=3)
    rec_names = catalog.set_index("Product_ID").loc[recs.index, "Product_Name"].tolist()
    risk = customer_context.loc[customer_context["Customer_ID"] == cid, "Churn_Risk_Score"].iloc[0]
    print(f"{cid} (churn risk {risk:.2f}): {rec_names}")

CUST_0214294 (churn risk 0.93): ['Premium Credit Card', 'Small Business Loan', 'Secured Credit Card']
CUST_0101511 (churn risk 0.92): ['Brokerage Account', 'Mutual Fund - Balanced', 'Retirement Account (IRA)']
CUST_0370674 (churn risk 0.92): ['Robo-Advisor Portfolio', 'Term Life Insurance', 'Homeowners Insurance']
CUST_0078600 (churn risk 0.92): ['Rewards Credit Card', 'Retirement Account (IRA)', 'Auto Loan']


CUST_0280178 (churn risk 0.92): ['Secured Credit Card', 'Home Mortgage', 'Auto Loan']


**Genuinely personalized, not a single generic "retention offer" repeated five times.** Each of
these five customers, all around a 92% predicted churn risk, gets a different recommendation
based on what they individually already hold: credit cards for the mortgage-only customer,
investment products for one whose existing holdings lean that way, insurance for another. That
variation is exactly the point, a one-size-fits-all retention offer ignores information this
system already has for free.

## <font color="brown">Business Insights and Recommendations</font>

- **At 450,000 customers, item-based collaborative filtering was the right structural choice**,
  not just a smaller version of user-based CF. Comparing 24 products to each other scales with
  the catalog, not the customer base, exactly the property a real bank's recommendation system
  needs.
- **Collaborative and content-based filtering can genuinely disagree**, and did, for a customer
  with only one product on file (cards vs. loans/savings). A thin history is precisely where a
  hybrid blend earns its keep, leaning on both signals instead of trusting either one alone.
- **The cold-start problem is structural, not a modeling choice**, confirmed at bank scale exactly
  as it was demonstrated at small scale earlier in this repo: a new customer gets a CF score of
  0.0 for everything, a new product gets no CF score at all, while content-based filtering handles
  both immediately from catalog features alone.
- **24,678 customers (5.5% of the base) sit at the true priority intersection**, high churn risk
  *and* at-risk behavior, confirmed independently by two different modeling approaches across
  Days 1 and 2. This is a business-actionable list, not just a modeling exercise: personalized,
  per-customer retention offers for exactly this group, generated automatically, at scale.
- **The three days build on each other for a reason.** Segmentation alone describes customers.
  Churn prediction alone flags risk without saying what to do about it. Only combining all three,
  segment, risk score, and recommendation, turns "this customer might leave" into "here is the
  specific offer most likely to keep them."